# GPAT

Gridded Plume Analysis Tool (GPAT) modelling framework. This simulates flight trajectories, estimates fuel burn and emissions, models dispersion effects, and aggregates plume data to a common Eulerian grid for further photochemical and microphysical processing.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from dataclasses import asdict
from pycontrails.models.gpat.gpat import GPAT, SimParams, FlParams, PlParams, MetParams, ChemParams, dict_to_dataclass
import os
import holoviews as hv
import hvplot.pandas
import hvplot.xarray

In [2]:
# global simulation parameters
sim_params = {
    "t_fl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=1)),# (start time, time step, run time)
    "t_pl": (pd.to_datetime("2022-01-20 13:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=2)),# (start time, time step, max age)
    "t_sim": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(seconds=20), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "t_out": (pd.to_datetime("2022-01-20 12:00:00"), pd.Timedelta(minutes=1), pd.Timedelta(hours=4)),# (start time, time step, run time)
    "lat_bounds": (0.0, 1.0),  # lat bounds [deg]
    "lon_bounds": (0.0, 1.0),  # lon bounds [deg]
    "alt_bounds": (10000, 11000),  # alt bounds [m]
    "hres_sim_c": 0.05,  # coarse horizontal resolution [deg]
    "vres_sim_c": 500,  # coarse vertical resolution [m]
    "hres_sim_f": 0.001,  # fine horizontal resolution [deg]
    "vres_sim_f": 100,  # fine vertical resolution [m]

    "run_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/",
    "data_path": "/home/ktait98/GPAT2026/pycontrails_kt/pycontrails/models/gpat/data/", # "/projects/Impact_of_aviation_on_climate
    "job_id": "GPAT_Feb_2026_test_2_ac",
}

In [3]:
#flight trajectory parameters
fl_params = {
    "mode": "synthetic",
    "file": None,  # flight trajectory file

    "ac_type": "A320",  # aircraft type
    "fl0_speed": 150.0,  # m/s
    "fl0_heading": 45.0,  # deg
    "fl0_coords0": (0.1, 0.1, 10500),  # lat, lon, alt [deg, deg, m]
    "sep_dist": (10000, 5000, 0),  # dx, dy, dz [m]
    "n_ac": 2,  # number of aircraft
}

In [4]:
# plume dispersion parameters
pl_params = {
    "depth": 50.0,  # initial plume depth, [m]
    "width": 50.0,  # initial plume width, [m]
    "verbose_outputs": False,  # print verbose outputs
    "shear": 0.01,  # shear [m/s]
    "n_slices": 3,  # number of slices in the plume
    "f_max": 0.99,  # maximum fraction of total emissions in any slice
    "output_pl_slices": True,  # output plume slices to netCDF
    "n_points": 16,  # number of points to represent the plume ellipse
    }

In [5]:
# meteorology parameters
met_params = {
    "eastward_wind": 5.0,  # m/s
    "northward_wind": 5.0,  # m/s
    "lagrangian_tendency_of_air_pressure": 0.0,  # m/s
}

In [6]:
# chemistry parameters
chem_params = {
    "run_chem": True,
    "species_emi": ("NO", "CO", "SO2"),
    # "species_pl": ("NO", "CO", "SO2"),
    "species_pl": ("NO", "NO2", "O3", "NO3", "N2O5",
                      "HNO3", "HONO", "HO2NO2","PAN", 
                      "CH3O2NO2","H2O2", "CH3OOH",
                      "CO", "CH4", "HCHO", "SO2", "SA"),
    "species_out": ("O3", "NO2", "NO", "NO3", "N2O5", 
                    "HNO3", "HONO", "HO2", "OH", "H2O2",
                    "CO", "CH4", "CH3O2","HO2NO2", "PAN", "SO2" )
}

In [7]:
sim_params = SimParams(**sim_params)
fl_params = FlParams(**fl_params)
pl_params = PlParams(**pl_params)
met_params = MetParams(**met_params)
chem_params = ChemParams(**chem_params)

gpat = GPAT(sim_params, fl_params, pl_params, met_params, chem_params)

In [10]:
from IPython.display import clear_output
clear_output(wait=True)

pl_ds = xr.open_dataset(f"{gpat.inputs_job}/pl_ds.nc")
pl_ds

<xarray.Dataset> Size: 772kB
Dimensions:          (seg_id: 30, time: 129, species_emi: 3)
Coordinates:
    flight_id        (seg_id) int64 240B ...
    waypoint         (seg_id) int64 240B ...
  * seg_id           (seg_id) int64 240B 1 2 3 4 5 6 7 ... 24 25 26 27 28 29 30
  * time             (time) <U20 10kB '2022-01-20T13:00:00Z' ... '2022-01-20T...
    species_emi_num  (species_emi) int64 24B ...
  * species_emi      (species_emi) <U3 36B 'NO' 'CO' 'SO2'
    active_seg_flag  (seg_id, time) int64 31kB ...
    time_rel_s       (time) int64 1kB ...
    time_idx         (time) int64 1kB ...
Data variables: (12/15)
    age              (seg_id, time) <U21 325kB ...
    longitude        (seg_id, time) float64 31kB ...
    latitude         (seg_id, time) float64 31kB ...
    level            (seg_id, time) float64 31kB ...
    width            (seg_id, time) float64 31kB ...
    depth            (seg_id, time) float64 31kB ...
    ...               ...
    sigma_zz         (seg_id, time) float64 31kB ...
    longitude_m      (seg_id, time) float64 31kB ...
    latitude_m       (seg_id, time) float64 31kB ...
    altitude         (seg_id, time) float64 31kB ...
    emi_pl_mass      (seg_id, species_emi) float64 720B ...
    age_s            (seg_id, time) int64 31kB ...
Attributes: (12/14)
    nseg:              30
    ts_fl:             60.0
    ts_pl:             60.0
    ts_sim:            20.0
    ts_out:            60.0
    species_emi:       ['NO', 'CO', 'SO2']
    ...                ...
    species_pl_num:    [  8   4   6   5   7  14  13  15 198 217  12 144  11  ...
    n_slices:          3
    f_max:             0.99
    output_pl_slices:  1
    n_points:          16
    description:       Emission species mass in plume segments

In [11]:
boxm_ds = xr.open_dataset(f"{gpat.inputs_job}/boxm_ds.nc")
boxm_ds

<xarray.Dataset> Size: 29MB
Dimensions:           (time: 721, cell: 800, species_boxm: 219)
Coordinates:
  * time              (time) <U20 58kB '2022-01-20T12:00:00Z' ... '2022-01-20...
    air_pressure      (cell) float64 6kB ...
    altitude_c        (cell) float64 6kB ...
  * species_boxm      (species_boxm) <U10 9kB 'O1D' 'O' 'OH' ... 'EMPOA' 'P2007'
    time_rel_s        (time) int64 6kB ...
    time_idx          (time) int64 6kB ...
    species_boxm_num  (species_boxm) int64 2kB ...
    level_c           (cell) float64 6kB ...
    longitude_c       (cell) float64 6kB ...
    latitude_c        (cell) float64 6kB ...
Dimensions without coordinates: cell
Data variables:
    air_temperature   (cell, time) float64 5MB ...
    H2O               (cell, time) float64 5MB ...
    M                 (cell, time) float64 5MB ...
    O2                (cell, time) float64 5MB ...
    N2                (cell, time) float64 5MB ...
    sza               (cell, time) float64 5MB ...
    Y_bg_c            (cell, species_boxm) float64 1MB ...
Attributes: (12/14)
    ts_fl:          60.0
    ts_pl:          60.0
    ts_sim:         20.0
    ts_out:         60.0
    hres_sim_c:     0.05
    vres_sim_c:     500
    ...             ...
    photol_params:  57
    photol_coeffs:  96
    therm_coeffs:   512
    flux_species:   130
    description:    BOXM coarse-grid meteorology and background chemistry fields
    note:           Emissions and plume segments handled separately via PL_DS...

In [12]:
pl_out = xr.open_dataset(f"{gpat.outputs_job}/pl_out.nc")
pl_out

<xarray.Dataset> Size: 6MB
Dimensions:         (seg_id: 30, time: 241, species_pl: 17, slice_id: 3,
                     pt_id: 16, coord: 3, corner_id: 4)
Coordinates:
    flight_id       (seg_id) int64 240B ...
    waypoint        (seg_id) int64 240B ...
  * seg_id          (seg_id) int64 240B 1 2 3 4 5 6 7 8 ... 24 25 26 27 28 29 30
  * time            (time) <U20 19kB '2022-01-20T12:00:00Z' ... '2022-01-20T1...
  * species_pl      (species_pl) <U8 544B 'NO' 'NO2' 'O3' ... 'HCHO' 'SO2' 'SA'
  * slice_id        (slice_id) int64 24B 1 2 3
  * pt_id           (pt_id) int64 128B 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16
  * coord           (coord) <U5 60B 'lon_m' 'lat_m' 'alt_m'
  * corner_id       (corner_id) <U2 32B 'BL' 'TL' 'TR' 'BR'
    species_pl_num  (species_pl) int64 136B ...
    time_rel_s      (time) int64 2kB ...
    time_idx        (time) int64 2kB ...
Data variables:
    y_half          (seg_id, slice_id, time) float64 174kB ...
    z_half          (seg_id, slice_id, time) float64 174kB ...
    m_frac          (slice_id) float64 24B ...
    w_slice         (slice_id) float64 24B ...
    ellipses_m      (seg_id, pt_id, coord, time) float64 3MB ...
    slice_polys_m   (seg_id, slice_id, corner_id, coord, time) float64 2MB ...
    pl_mass         (seg_id, species_pl, time) float64 983kB ...
Attributes:
    description:  Plume segment output for BOXM

In [13]:
from IPython.display import clear_output
clear_output(wait=True)

pl_out = xr.open_dataset(f"{gpat.outputs_job}/pl_out.nc")
pl_out

gpat.analysis.load_output_datasets()

In [14]:
gpat.analysis.plot_plumes_3d_pv(time_idx=250)

[]
[]
[]
[]


ValueError: Unsupported key-type <class 'xarray.core.dataarray.DataArray'>

In [ ]:
pd.DataFrame({
    "species_emi": pl_ds["species_emi"].values,
    "species_emi_num": pl_ds["species_emi_num"].values,
})

pd.DataFrame({
    "species_pl": pl_out["species_pl"].values,
    "species_pl_num": pl_out["species_pl_num"].values,
})

print(pd.DataFrame({
    "species_emi": pl_ds["species_emi"].values,
    "species_emi_num": pl_ds["species_emi_num"].values,
}).to_string(index=False))

print(pd.DataFrame({
    "species_pl": pl_out["species_pl"].values,
    "species_pl_num": pl_out["species_pl_num"].values,
}).to_string(index=False))

species_emi  species_emi_num
         NO                8
         CO               11
        SO2               16
species_pl  species_pl_num
        NO               8
       NO2               4
        O3               6
       NO3               5
      N2O5               7
      HNO3              14
      HONO              13
    HO2NO2              15
       PAN             198
  CH3O2NO2             217
      H2O2              12
    CH3OOH             144
        CO              11
       CH4              21
      HCHO              39
       SO2              16
        SA              20


In [ ]:
from IPython.display import clear_output
clear_output(wait=True)

boxm_out = xr.open_dataset(f"{gpat.outputs_job}/boxm_out.nc")

boxm_out

<xarray.Dataset> Size: 50MB
Dimensions:          (cell: 800, species_out: 16, time: 241)
Coordinates:
  * time             (time) <U20 19kB '2022-01-20T12:00:00Z' ... '2022-01-20T...
    time_rel_s       (time) int64 2kB ...
    time_idx         (time) int64 2kB ...
    altitude_c       (cell) float64 6kB ...
  * species_out      (species_out) <U6 384B 'O3' 'NO2' 'NO' ... 'PAN' 'SO2'
    species_out_num  (species_out) int64 128B ...
    level_c          (cell) float64 6kB ...
    longitude_c      (cell) float64 6kB ...
    latitude_c       (cell) float64 6kB ...
Dimensions without coordinates: cell
Data variables:
    Y_bg_c           (cell, species_out, time) float64 25MB ...
    Y_del_c          (cell, species_out, time) float64 25MB ...
    active_flag      (cell, time) bool 193kB ...
Attributes:
    description:  BOXM output coarse-grid chemistry fields